# Protein Analysis Workflow

Two common starting points for protein TMAP:

- **Part A** -- You have a FASTA file and want to explore sequence properties.  
  No external dependencies beyond TMAP.
- **Part B** -- You have pre-computed embeddings (ESM, ESM-2, ProtTrans, etc.) and want
  to build a large-scale map with AlphaFold / UniProt annotations.
  Requires pre-computed embedding files.
- **Part C** -- You have all-vs-all alignment results (MMseqs2, Foldseek, BLAST)
  and want to build a TMAP from sequence or structure similarity.

**Requirements:** `pip install tmap`

---
## Part A: From FASTA to TMAP

The most common starting point -- you have a FASTA file of protein sequences.

TMAP provides three file readers in `tmap.utils`:

| Format | Function | Input |
|--------|----------|-------|
| FASTA (.fa, .fasta) | `read_fasta(path)` | Sequences + headers |
| CSV / TSV | `read_protein_csv(path, id_col, seq_col)` | Tabular data |
| ID list (.txt) | `read_id_list(path)` | One UniProt accession per line |

### 1. Load sequences from FASTA

In [ ]:
import numpy as np
from tmap.utils import read_fasta

# SCOPe ASTRAL 40% subset -- protein domains with structural classification
ids_fasta, sequences = read_fasta("../examples/data/scope_cache/astral_40.fa", max_seqs=50_000)

print(f"{len(sequences)} sequences loaded")
print(f"Length range: {min(len(s) for s in sequences)} -- {max(len(s) for s in sequences)} aa")
print(f"First ID:  {ids_fasta[0]}")
print(f"First seq: {sequences[0][:60]}...")

The other readers work the same way:

```python
# CSV/TSV with header row
ids, seqs = read_protein_csv("proteins.csv", id_col="accession", seq_col="sequence")

# Plain text ID list (one per line, # comments allowed)
ids = read_id_list("uniprot_ids.txt")
```

### 2. Compute sequence properties

`sequence_properties` computes physicochemical descriptors from amino acid
sequences -- no external dependencies, no network access.

In [ ]:
from tmap.utils import sequence_properties

seq_props = sequence_properties(sequences)

for key, values in seq_props.items():
    print(f"{key:25s}  min={np.nanmin(values):8.1f}  max={np.nanmax(values):10.1f}")

### 3. Build a TMAP from sequences

For a quick exploration without pre-computed embeddings, we can encode
sequences as **k-mer frequency vectors** and use cosine distance.
This is fast and dependency-free -- a good first pass before investing in
ESM-2 embeddings.

In [ ]:
from collections import Counter

def kmer_features(sequences, k=3):
    """Encode sequences as k-mer frequency vectors."""
    all_kmers = set()
    counters = []
    for seq in sequences:
        seq = seq.upper()
        c = Counter(seq[i:i+k] for i in range(len(seq) - k + 1))
        counters.append(c)
        all_kmers.update(c.keys())

    kmer_list = sorted(all_kmers)
    kmer_idx = {km: i for i, km in enumerate(kmer_list)}
    X = np.zeros((len(sequences), len(kmer_list)), dtype=np.float32)
    for i, c in enumerate(counters):
        total = sum(c.values())
        for km, count in c.items():
            X[i, kmer_idx[km]] = count / total
    return X

X = kmer_features(sequences, k=3)
print(f"Feature matrix: {X.shape}  (3-mer frequencies)")

In [ ]:
from tmap import TMAP

model_fasta = TMAP(metric="cosine", n_neighbors=15, seed=42).fit(X)

print(f"Embedding: {model_fasta.embedding_.shape}")
print(f"Tree edges: {model_fasta.tree_.edges.shape[0]}")

### 4. Sequence properties as color layers

Use the estimator's built-in `plot()` to explore sequence properties
interactively. Pass a DataFrame to `data=` and select which column
to color by.

In [ ]:
import re
import pandas as pd

# Parse SCOP class from headers (a=all-alpha, b=all-beta, c=a/b, d=a+b, ...)
scop_class_names = {
    "a": "All-alpha", "b": "All-beta", "c": "Alpha/beta",
    "d": "Alpha+beta", "e": "Multi-domain", "f": "Membrane", "g": "Small",
}
scop_re = re.compile(r"\s([a-g])\.\d+\.\d+\.\d+\s")
classes = []
for h in ids_fasta:
    m = scop_re.search(h)
    classes.append(scop_class_names.get(m.group(1), "other") if m else "other")

props_df = pd.DataFrame(seq_props)
props_df["scop_class"] = classes

model_fasta.plot(
    color_by="gravy",
    data=props_df,
    tooltip_properties=["length", "molecular_weight", "isoelectric_point"],
    point_size=2,
)

### 5. TmapViz export

Pass the sequence properties dict directly to `add_color_layout` --
it auto-detects continuous vs. categorical for each column.

In [ ]:
viz = model_fasta.to_tmapviz()
viz.title = "SCOPe Protein Domains -- 3-mer TMAP"

# Bulk-add all sequence properties as color layers
for key in seq_props.keys():
    viz.add_color_layout(key, seq_props[key])

viz.add_color_layout("SCOP class", classes, categorical=True, color="Set2")

print(f"Layouts: {[l.name for l in viz.layouts]}")
viz.show()

In [ ]:
model_fasta.plot_static(color_by=classes, color_map="Set2", point_size=3)

---
## Part B: Pre-computed embeddings + API annotations

For production protein TMAPs, you typically have:

1. **Embeddings** from ESM-2, ProtTrans, or similar (`.npy` files)
2. **UniProt IDs** for each protein

We load 49k ESM-2 embeddings and annotate them with AlphaFold structural
metadata and UniProt annotations.

ESM-2 embeddings can be generated with:
- `esm` Python package (Meta)
- ESM Metagenomic Atlas API
- Local inference with `fair-esm`

### 5. Load embeddings

This assumes pre-computed embeddings saved as NumPy arrays.
The paths `../data/embeddings.npy` and `../data/ids.npy` are dataset-specific --
replace them with your own.

In [ ]:
embeddings = np.load("../data/embeddings.npy").astype(np.float32)
ids = np.load("../data/ids.npy", allow_pickle=True)

print(f"Embeddings: {embeddings.shape} ({embeddings.dtype})")
print(f"IDs: {ids.shape}")
print(f"Sample IDs: {ids[:5]}")

### 6. Fit TMAP on embeddings

Euclidean is the recommended metric for ESM-2 embeddings.
USearch handles the dense nearest-neighbor search automatically.

In [ ]:
# Euclidean is the recommended for ESM embeddings. Cosine should work as well
model = TMAP(metric="euclidean", n_neighbors=20, seed=42).fit(embeddings)

print(f"Embedding: {model.embedding_.shape}")
print(f"Tree edges: {model.tree_.edges.shape[0]}")

### 7. Fetch AlphaFold structural metadata

`fetch_alphafold` queries the AlphaFold DB REST API -- one request per protein,
threaded with 20 workers. For 49k proteins this takes a few minutes.

Returns `float32` arrays with NaN for proteins not in AlphaFold DB (~14% missing).

In [ ]:
from tmap.utils import fetch_alphafold

af_data = fetch_alphafold(ids.tolist())

for key, values in af_data.items():
    n_ok = np.sum(~np.isnan(values))
    print(f"{key:20s}  {n_ok:,}/{len(values):,} fetched  "
          f"mean={np.nanmean(values):.2f}")

### 8. Fetch UniProt annotations

`fetch_uniprot` batch-fetches text annotations (organism, GO terms,
subcellular location, etc.) via the UniProt REST API.

In [ ]:
from tmap.utils import fetch_uniprot

up_data = fetch_uniprot(ids.tolist())

# Show a few
for i in range(5):
    name = str(up_data["protein_name"][i])[:40]
    org = str(up_data["organism_name"][i])[:30]
    print(f"  {ids[i]:12s}  {name:40s}  {org}")

### 9. Visualize with `add_metadata`

Now we combine everything into a single visualization. `add_metadata`
auto-detects continuous vs categorical -- pass the AlphaFold dict directly.

In [ ]:
viz = model.to_tmapviz()
viz.title = "Protein Embedding Space -- 49k ESM-2"

# AlphaFold structural metrics as color layers
viz.add_color_layout("pLDDT", af_data["plddt"].tolist())
viz.add_color_layout("Fraction confident", af_data["frac_confident"].tolist())
viz.add_color_layout("Fraction disordered", af_data["frac_disordered"].tolist())

# UniProt annotation score as a color layer
viz.add_color_layout("Length", up_data["length"].tolist())
viz.add_color_layout("annotation_score", up_data["annotation_score"].tolist(),
                     categorical=False, color="viridis")

# UniProt text fields as tooltip labels
viz.add_label("Subcellular Location", up_data["cc_subcellular_location"].tolist())
viz.add_label("Protein", list(up_data["protein_name"]))
viz.add_label("Organism", list(up_data["organism_name"]))

# Clickable UniProt links + AlphaFold 3D structure viewer
viz.add_protein_ids(ids.tolist())

print(f"Color layers: {[l.name for l in viz.layouts]}")

In [ ]:
# viz.write_html("protein_map.html")

In [ ]:
model.plot_static(color_by=af_data["plddt"], color_map="RdYlGn", point_size=1)

### 10. Tree exploration

Pick a high-confidence protein and trace pseudotime from it.

In [ ]:
# Find the protein with highest pLDDT
best = int(np.nanargmax(af_data["plddt"]))
print(f"Root: {ids[best]} (pLDDT = {af_data['plddt'][best]:.1f})")

pseudotime = model.distances_from(best)
print(f"Pseudotime range: [{pseudotime.min():.1f}, {pseudotime.max():.1f}]")

model.plot_static(color_by=pseudotime, color_map="magma", point_size=1)

### 11. Caching fetched data

`fetch_alphafold` and `fetch_uniprot` are stateless -- they don't cache
results. For large datasets, save the results yourself:

In [ ]:
# Save
# np.savez("alphafold_cache.npz", **af_data)

# Load later
# cached = np.load("alphafold_cache.npz")
# af_data = {k: cached[k] for k in cached.files}

---
## Part C: Alignment-based TMAP

When you have all-vs-all alignment results from MMseqs2, Foldseek, or BLAST,
you can build a TMAP directly from pairwise similarity scores using
`parse_alignment`. This avoids embedding entirely -- the alignment scores
*are* the similarity metric.

Input: standard BLAST6 / m8 tabular format (12 tab-separated columns):
```
qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore
```

In [ ]:
from tmap.utils import parse_alignment

# Load precomputed all-vs-all search (BLAST6 / m8 format)
# Replace with your own alignment file
# knn, id_order = parse_alignment("hits.m8", k=20, score_col="bitscore")

# Fit TMAP with precomputed KNN from alignment scores
# model_aln = TMAP(metric="precomputed").fit(knn_graph=knn)
# print(f"Embedding: {model_aln.embedding_.shape}")
# print(f"Proteins: {len(id_order)}")

`parse_alignment` reads the m8 file, keeps the top-k hits per query
ranked by `score_col` (default: `bitscore`), and returns a `KNNGraph`
compatible with `TMAP(metric="precomputed")`.

This works with:
- **MMseqs2** `easy-search` or `search` output
- **Foldseek** `easy-search` (structure similarity via 3Di + TM-score)
- **BLAST** `-outfmt 6` tabular output

Use `score_col="evalue"` for E-value-based ranking (lower = better;
`parse_alignment` handles the inversion automatically via `as_distance=True`).

## Summary

### File readers

| Format | Code |
|--------|------|
| FASTA | `ids, seqs = read_fasta("proteins.fa")` |
| CSV/TSV | `ids, seqs = read_protein_csv("data.csv", id_col="accession", seq_col="sequence")` |
| ID list | `ids = read_id_list("uniprot_ids.txt")` |
| Alignment (m8) | `knn, ids = parse_alignment("hits.m8", k=20)` |

### Part A: FASTA workflow

| Step | Code |
|------|------|
| Parse FASTA | `ids, seqs = read_fasta("proteins.fa")` |
| Sequence properties | `props = sequence_properties(seqs)` |
| Encode (k-mers) | `X = kmer_features(seqs, k=3)` |
| Fit TMAP | `model = TMAP(metric="cosine").fit(X)` |
| Interactive plot | `model.plot(color_by="gravy", data=props_df)` |
| TmapViz export | `viz.add_color_layout(key, props[key])` |

For higher quality, replace k-mers with ESM-2 embeddings
(see `examples/protein_fold_space.py`).

### Part B: Embeddings + API annotations

| Step | Code |
|------|------|
| Load embeddings | `emb = np.load("embeddings.npy")` |
| Fit TMAP | `model = TMAP(metric="euclidean").fit(emb)` |
| AlphaFold metadata | `af = fetch_alphafold(ids)` |
| UniProt annotations | `up = fetch_uniprot(ids)` |
| Structural metrics | `viz.add_color_layout("pLDDT", af["plddt"])` |
| Annotation score | `viz.add_color_layout("score", up["annotation_score"])` |
| Tooltip labels | `viz.add_label("Protein", up["protein_name"])` |
| 3D structure viewer | `viz.add_protein_ids(ids)` |
| Cache results | `np.savez("af_cache.npz", **af)` |

### Part C: Alignment-based

| Step | Code |
|------|------|
| Parse alignments | `knn, ids = parse_alignment("hits.m8", k=20)` |
| Fit TMAP | `model = TMAP(metric="precomputed").fit(knn_graph=knn)` |